In [ ]:
import subprocess
import sys

print("Installing packages...")
subprocess.run(["pip", "install", "nerfstudio"], check=True, capture_output=True)
subprocess.run(["pip", "install", "opencv-python", "imageio", "imageio-ffmpeg", "scikit-image"],
               check=True, capture_output=True)
subprocess.run(["apt-get", "update"], check=True, capture_output=True)
subprocess.run(["apt-get", "install", "-y", "colmap"], check=True, capture_output=True)

print("Installation complete!")

Installing packages...
Installation complete!


In [ ]:
import os
import cv2
from google.colab import files
import shutil

# Create folders
os.makedirs("project_data/raw_video", exist_ok=True)
os.makedirs("project_data/frames/downsampled", exist_ok=True)
os.makedirs("project_data/colmap_input/images", exist_ok=True)

# Upload video
print("Upload your video file:")
uploaded = files.upload()
video_file = list(uploaded.keys())[0]

# Move to correct folder
video_path = f"project_data/raw_video/{video_file}"
shutil.move(video_file, video_path)

# Check video info
cap = cv2.VideoCapture(video_path)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

print(f"\nVideo Info:")
print(f"  Frames: {frame_count}")
print(f"  FPS: {fps}")
print(f"  Resolution: {width}x{height}")
print(f"  Duration: {frame_count/fps:.1f} seconds")

Upload your video file:


Saving 1000133350.mp4 to 1000133350.mp4

Video Info:
  Frames: 6921
  FPS: 60.004456334425065
  Resolution: 1080x1920
  Duration: 115.3 seconds


In [ ]:
def extract_frames(video_path, output_dir, frame_interval=10, target_height=480):
    """Extract every nth frame and resize"""
    cap = cv2.VideoCapture(video_path)
    frame_id = 0
    extracted_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_id % frame_interval == 0:
            h, w = frame.shape[:2]
            aspect_ratio = w / h
            new_width = int(target_height * aspect_ratio)
            frame_resized = cv2.resize(frame, (new_width, target_height))

            output_path = os.path.join(output_dir, f"frame_{extracted_count:06d}.jpg")
            cv2.imwrite(output_path, frame_resized)
            extracted_count += 1

            if extracted_count % 50 == 0:
                print(f"  Extracted {extracted_count} frames...")

        frame_id += 1

    cap.release()
    return extracted_count

# Extract frames
print("Extracting frames...")
num_frames = extract_frames(
    video_path=video_path,
    output_dir="project_data/frames/downsampled",
    frame_interval=10,
    target_height=480
)

print(f"✓ Extracted {num_frames} frames at 480p")

# Copy to COLMAP folder
colmap_images = "project_data/colmap_input/images"
for frame_file in sorted(os.listdir("project_data/frames/downsampled")):
    src = os.path.join("project_data/frames/downsampled", frame_file)
    dst = os.path.join(colmap_images, frame_file)
    shutil.copy(src, dst)

print(f"Copied frames to COLMAP folder")

Extracting frames...
  Extracted 50 frames...
  Extracted 100 frames...
  Extracted 150 frames...
  Extracted 200 frames...
  Extracted 250 frames...
  Extracted 300 frames...
  Extracted 350 frames...
  Extracted 400 frames...
  Extracted 450 frames...
  Extracted 500 frames...
  Extracted 550 frames...
  Extracted 600 frames...
  Extracted 650 frames...
✓ Extracted 693 frames at 480p
Copied frames to COLMAP folder


In [ ]:
import subprocess

colmap_input = "project_data/colmap_input/images"
colmap_db = "project_data/colmap.db"
colmap_output = "project_data/colmap_output"

os.makedirs(colmap_output, exist_ok=True)

print("Running COLMAP (this takes 10-30 minutes)...\n")

# Feature extraction
print("[1/3] Feature extraction...")
subprocess.run([
    "colmap", "feature_extractor",
    "--database_path", colmap_db,
    "--image_path", colmap_input
], check=True)

# Feature matching
print("[2/3] Feature matching...")
subprocess.run([
    "colmap", "exhaustive_matcher",
    "--database_path", colmap_db
], check=True)

# Mapper
print("[3/3] Sparse reconstruction...")
subprocess.run([
    "colmap", "mapper",
    "--database_path", colmap_db,
    "--image_path", colmap_input,
    "--output_path", colmap_output
], check=True)

print("\n COLMAP complete!")

Running COLMAP (this takes 10-30 minutes)...

[1/3] Feature extraction...


CalledProcessError: Command '['colmap', 'feature_extractor', '--database_path', 'project_data/colmap.db', '--image_path', 'project_data/colmap_input/images']' died with <Signals.SIGABRT: 6>.

In [ ]:
from nerfstudio.scripts.colmap_to_json import colmap_to_json
from pathlib import Path
import json

# Create nerfstudio data directory
nerfstudio_data = "project_data/nerfstudio_data"
os.makedirs(nerfstudio_data, exist_ok=True)

# Move images to nerfstudio format
images_dir = f"{nerfstudio_data}/images"
os.makedirs(images_dir, exist_ok=True)

for frame_file in sorted(os.listdir("project_data/frames/downsampled")):
    src = os.path.join("project_data/frames/downsampled", frame_file)
    dst = os.path.join(images_dir, frame_file)
    shutil.copy(src, dst)

print(f"✓ Copied {len(os.listdir(images_dir))} frames to nerfstudio format")

# Run COLMAP via nerfstudio
print("\nRunning COLMAP via nerfstudio...")
subprocess.run([
    "ns-process-data",
    "colmap",
    "--data", nerfstudio_data,
    "--colmap-model-type", "sequential"
], check=True)

print("✓ COLMAP processing complete!")

ModuleNotFoundError: No module named 'nerfstudio.scripts.colmap_to_json'

In [ ]:
nerfstudio_data = "project_data/nerfstudio_data"
os.makedirs(nerfstudio_data, exist_ok=True)

# Create images folder
images_dir = f"{nerfstudio_data}/images"
os.makedirs(images_dir, exist_ok=True)

# Copy frames
for frame_file in sorted(os.listdir("project_data/frames/downsampled")):
    src = os.path.join("project_data/frames/downsampled", frame_file)
    dst = os.path.join(images_dir, frame_file)
    shutil.copy(src, dst)

print(f"Prepared {len(os.listdir(images_dir))} frames for training")

# Let nerfstudio handle camera estimation automatically
print("Ready for NeRF training (nerfstudio will estimate cameras)")

Prepared 693 frames for training
Ready for NeRF training (nerfstudio will estimate cameras)


In [ ]:
import time

print("Training NeRF model")

nerf_start = time.time()

result = subprocess.run([
    "ns-train",
    "nerfacto",
    "--data", nerfstudio_data,
    "--output-dir", "project_data/nerf_output",
    "--max-num-iterations", "20000",
    "--vis", "none"
], capture_output=False, text=True)

nerf_time = time.time() - nerf_start

print(f"\nNeRF training complete in {nerf_time/60:.1f} minutes")

Training NeRF model

NeRF training complete in 0.7 minutes


In [ ]:
# Check if output was created
import os

if os.path.exists("project_data/nerf_output"):
    print("NeRF output folder exists")
    print(os.listdir("project_data/nerf_output"))
else:
    print("❌ NeRF output folder NOT created - training failed")

# Check nerfstudio data
print("\nNerfstudio data:")
print(os.listdir(nerfstudio_data))

❌ NeRF output folder NOT created - training failed

Nerfstudio data:
['images']


In [ ]:
# Use video input directly (simpler for nerfstudio)
video_input = "project_data/raw_video"

print("Training NeRF directly from video...")
print("(nerfstudio will handle camera estimation automatically)\n")

import time
nerf_start = time.time()

result = subprocess.run([
    "ns-train",
    "nerfacto",
    "video-data",
    "--data", video_path,  # Use your original video
    "--output-dir", "project_data/nerf_output",
    "--max-num-iterations", "15000"
])

nerf_time = time.time() - nerf_start
print(f"\n✓ Training complete in {nerf_time/60:.1f} minutes")

Training NeRF directly from video...
(nerfstudio will handle camera estimation automatically)


✓ Training complete in 0.5 minutes


In [ ]:
# Check if nerfstudio is installed
result = subprocess.run(["which", "ns-train"], capture_output=True, text=True)
print("ns-train location:", result.stdout)
print("Error (if any):", result.stderr)

# Try importing nerfstudio directly
try:
    import nerfstudio
    print("nerfstudio imported successfully")
    print("Version:", nerfstudio.__version__)
except ImportError as e:
    print("nerfstudio import failed:", e)

# List what we have
result = subprocess.run(["pip", "list"], capture_output=True, text=True)
if "nerfstudio" in result.stdout:
    print("nerfstudio is in pip list")
else:
    print("nerfstudio NOT in pip list")

ns-train location: /usr/local/bin/ns-train

Error (if any): 
nerfstudio imported successfully


AttributeError: module 'nerfstudio' has no attribute '__version__'

In [ ]:
# Use nerfstudio's instant-ngp (faster, simpler)
print("Training with instant-ngp (faster model)...\n")

import time
start = time.time()

result = subprocess.run([
    "ns-train",
    "instant-ngp",
    "images",
    "--data", "project_data/nerfstudio_data",
    "--output-dir", "project_data/nerf_output",
    "--max-num-iterations", "10000",
    "--machine.num-gpus", "1"
], text=True)

elapsed = time.time() - start
print(f"\nTraining finished in {elapsed/60:.1f} minutes")

# Check if it worked
if os.path.exists("project_data/nerf_output"):
    print("Output created")
    print(os.listdir("project_data/nerf_output"))
else:
    print("Still failed")

Training with instant-ngp (faster model)...


Training finished in 0.5 minutes
Still failed


In [ ]:
# Try using nerfstudio's web interface approach with proper initialization
subprocess.run(["ns-install-cli"], capture_output=True)

print("Checking nerfstudio installation...")
subprocess.run(["python", "-m", "nerfstudio.scripts.train"], check=False)

Checking nerfstudio installation...


CompletedProcess(args=['python', '-m', 'nerfstudio.scripts.train'], returncode=2)

In [ ]:
# Use Luma AI's open-source NeRF tool (much simpler)
print("Installing luma...")
subprocess.run(["pip", "install", "torch", "torchvision", "pytorch3d", "-q"], check=True)

print("Ready for training")

Installing luma...


CalledProcessError: Command '['pip', 'install', 'torch', 'torchvision', 'pytorch3d', '-q']' returned non-zero exit status 1.